In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

/Users/evan/Projects/llm-zoomcamp-2026-code/module_02_vector_search/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7591.43it/s]


In [23]:
q0 = "Can I still join the course after the start date?"
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"
q3 = "How to install Docker on Windows?"

In [24]:
v0 = model.encode(q0)
v1 = model.encode(q1)
v2 = model.encode(q2)
v3 = model.encode(q3)

In [25]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [26]:
for v in [v0, v1, v2, v3]:
    print(v.dot(dv)) 

0.3233239
0.39572877
0.46365494
0.019730594


## Embedding Our Dataset

In [8]:
from ingest import load_faq_data

documents = load_faq_data()

In [9]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [10]:
# We concatenate ALL our questions and answers into a list of lists 
texts = [document['question'] + ' ' + document['answer'] for document in documents]

In [11]:
# for example, the question and answer from document 10 above is now one long string

print(f"question: {documents[10].get('question')}")
print(f"answer: {documents[10].get('answer')}")
texts[10]

question: Course: How many hours per week am I expected to spend on this course?
answer: It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.

You can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.


'Course: How many hours per week am I expected to spend on this course? It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'

In [12]:
len(texts)

1349

In [13]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)


len(vectors)

100%|██████████| 27/27 [00:07<00:00,  3.65it/s]


1349

## Implement Vector Search

In [ ]:
import numpy as np 

#Convert the vectors into an numpy array for lightening fast processing
X = np.array(vectors)

In [15]:
#Still the same shape as above: we have the exact same number of documents
len(X)

1349

In [16]:
#Now we compute the similiarity from our first question above: 
print(q1)
scores = X.dot(v1)

I just discovered the course, can I still join?


In [17]:
#np.argmax finds the highest score. 
idx = np.argmax(scores)

#the code below returns the index for the highest score
idx, scores[idx]

(np.int64(538), np.float32(0.8317791))

In [18]:
#So, we can pass the index to our knowledge base to identify the correct document
documents[idx]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [19]:
#optionally, we can sort and take the top n 

#top 5
top_5 = np.argsort(scores)[-5:]

#The elements are sorted from smallest to largest so we'll reverse it 
top_5 = top_5[::-1] 
top_5

array([538, 906, 624,   2, 503])

In [20]:
scores[top_5]

array([0.8317791 , 0.6845697 , 0.61755615, 0.60880876, 0.58479655],
      dtype=float32)

In [ ]:
for idx in top_5: 
    print(scores[idx])
    print(documents[idx])
    print()
 

0.8317791
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

0.6845697
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'}

0.61755615
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 's

## Import Vecttor Search
But why bother doing all that ourselves when we can just import a package which does it for us AND allows us to filter out certain fields? 

In [ ]:

from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [31]:
query = "I just discovered the course. Can I still join it?"

query_vector = model.encode(query)

results = vindex.search(query_vector=query_vector, num_results=5, filter_dict={'course':'llm-zoomcamp'})

In [32]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

## RAG with Vector Search

All learning is repetition so, let's review the three steps of RAG: 

1. Search for the user's query
2. Generate a response based on the query AND our knowledge base
3. Return the response to the user

In [34]:
from openai import OpenAI
openai_client = OpenAI()

In [35]:
# Now we need to download and index the data 

from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents=documents)

In [36]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client
)

In [37]:
query = "I just found out about the program, can I still sign up?"

assistant.rag(query=query)

'Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

### Switching to Vector Search 

The example above is still using text search; we need to swap out our search function to implement vector search. 

To do so, we make use of Object Oriented Programming principles: we create a subclass of RAGBase like this: 

In [40]:
class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course':self.course}

        vector_search_results = self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

        return vector_search_results

In [48]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client
)

In [49]:
query = 'I just found out about the program, can I still sign up?'

In [50]:
vector_assistant.rag(query=query)

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

## Vector Search with sqlitesearch

Rather than computing the dot product for EVERY document we have, we can implement 'Approximate Nearest Neighbours' (ANN) to limit the number of documents we actually search. 

We're also going to use Alex's `sqlitesearch` to persist our vectors. 

Basically, we'll save our knowledge base in a database. 

In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf', ## How we are using ANN; i.e., which approach
    db_path='faq_vectors2.db'
)

In [56]:
vs_index.fit(vectors=vectors, payload=documents)

In [60]:
query = "I just discovered the course. Can I still join it?"

query_vector = model.encode(inputs=query)

results = vs_index.search(query_vector=query_vector, filter_dict={'course':'llm-zoomcamp'}, num_results=5)

In [61]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [62]:
# Close the connection to the database so we can open it from somewhere else. 
vs_index.close()